# 🕊️ Make your Christian-AI training data — no terminal needed

This notebook does **Phase 1** for you. It turns the CAB-FF benchmark into
training data and downloads two files to your computer.

## You only do 3 things:
1. ✏️ In **STEP 1** below, paste your API key into the box.
2. 🔽 Pick `anthropic` (for a Claude `sk-ant-...` key) or `openai`.
3. ▶️ Top menu: **Runtime → Run all**. Then wait ~30–45 min.

When it finishes, **two files download automatically** — `train.jsonl` and
`eval.jsonl`. Those are what you upload to AutoTrain in Phase 2.

> ⚠️ **This costs money on your API key.** The default teacher is
> **Claude Opus 4.8** (top quality) ≈ **$35–70**. If you'd rather spend
> ~**$3–5**, use an OpenAI key and pick `openai` (uses gpt-4o-mini —
> lower quality but totally usable). Make sure your account has credit.

> 💡 Colab Free is fine here — this step uses **no GPU**, just API calls.
> Colab Pro only helps by keeping the tab alive longer. Keep the tab open.

In [ ]:
#@title ✏️ STEP 1 — paste your API key, then pick the type

MY_KEY = "PASTE-YOUR-KEY-HERE"  #@param {type:"string"}

# Where did your key come from?
#   anthropic = console.anthropic.com (sk-ant-...)  -> teacher = Claude Opus 4.8 (best)
#   openai    = platform.openai.com   (sk-...)      -> teacher = gpt-4o-mini (cheapest)
KEY_TYPE = "anthropic"  #@param ["anthropic", "openai"]

In [ ]:
#@title ▶️ STEP 2 — run everything (Runtime → Run all, or press play here)

import os, sys, subprocess, shutil

assert MY_KEY and "PASTE" not in MY_KEY, \
    "⛔ Go back to STEP 1 and paste your real API key first!"

print("⏳ (1/4) Downloading the project...")
if os.path.isdir("/content/SoliDeoGloria"):
    shutil.rmtree("/content/SoliDeoGloria")  # clear any half-finished earlier run
subprocess.run(["git", "clone", "--quiet",
                "https://github.com/moonshineaitech/SoliDeoGloria",
                "/content/SoliDeoGloria"], check=True)
os.chdir("/content/SoliDeoGloria/trainer")

print("⏳ (2/4) Installing the data tools (~2 min, lots of output is normal)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[teachers]", "-q"], check=True)

if KEY_TYPE == "anthropic":
    os.environ["ANTHROPIC_API_KEY"] = MY_KEY
    TEACHER = "claude-opus-4-8"
else:
    os.environ["OPENAI_API_KEY"] = MY_KEY
    TEACHER = "gpt-4o-mini"

print(f"⏳ (3/4) Writing training data with teacher = {TEACHER} (~30-45 min, be patient)...")
print("      (per-example API hiccups are skipped automatically; let it run.)\n")
proc = subprocess.run([sys.executable, "-m", "trainer.data.pipeline.cli", "build",
                       "--dataset", "../data/CAB_FF_v3_dataset.json",
                       "--teacher", TEACHER,
                       "--max-synth", "800", "--max-prefs", "400",
                       "--out", "data/built"])
if proc.returncode != 0:
    raise SystemExit("\n❌ Data build failed. Read the error printed ABOVE this "
                     "line and send it to Claude — don't just send this message.")

print("\n⏳ (4/4) Checking the files and sending them to your Downloads folder...")
n_train = sum(1 for _ in open("data/built/train.jsonl"))
n_eval = sum(1 for _ in open("data/built/eval.jsonl"))
print(f"      train.jsonl = {n_train} examples   |   eval.jsonl = {n_eval} examples")
assert n_train > 50, ("train.jsonl came out too small — something is wrong. "
                      "Send this whole output to Claude.")

from google.colab import files
for f in ["train.jsonl", "eval.jsonl", "pref_train.jsonl", "pref_eval.jsonl"]:
    path = f"data/built/{f}"
    if os.path.exists(path):
        files.download(path)

print("\n✅ DONE! Check your browser's Downloads folder.")
print("   Next: upload train.jsonl + eval.jsonl to AutoTrain (Phase 2 in AUTOTRAIN.md).")

## 🆘 If something goes wrong

| What you see | What to do |
|---|---|
| Stops on STEP 1 with "paste your real key" | You didn't replace `PASTE-YOUR-KEY-HERE`. Edit STEP 1, run again. |
| `❌ Data build failed` | Scroll up to the red error ABOVE that line and send **that** text to Claude. |
| An error mentioning `rate limit` / `insufficient quota` / `credit balance` | Add credit to your API account, then **Runtime → Run all** again. |
| The tab disconnected | Reopen it and **Runtime → Run all** again — it starts clean. |
| No download popped up | Click the 📁 folder icon on the left → `SoliDeoGloria/trainer/data/built/` → download `train.jsonl` and `eval.jsonl` by hand. |

*Soli Deo Gloria.*